In [2]:
import torch
import torch.nn as nn

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
df = pd.read_csv('Data/NYCTaxiFares.csv')

In [4]:
df.head()

,pickup_datetime,fare_amount,fare_class,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2010-04-19 08:17:56 UTC,6.5,0,-73.992365,40.730521,-73.975499,40.744746,1
1,2010-04-17 15:43:53 UTC,6.9,0,-73.990078,40.740558,-73.974232,40.744114,1
2,2010-04-17 11:23:26 UTC,10.1,1,-73.994149,40.751118,-73.960064,40.766235,2
3,2010-04-11 21:25:03 UTC,8.9,0,-73.990485,40.756422,-73.971205,40.748192,1
4,2010-04-17 02:19:01 UTC,19.7,1,-73.990976,40.734202,-73.905956,40.743115,1


In [5]:
df['fare_amount'].describe()

count    120000.000000
mean         10.040326
std           7.500134
min           2.500000
25%           5.700000
50%           7.700000
75%          11.300000
max          49.900000
Name: fare_amount, dtype: float64

In [6]:
def haversine_distance(df, lat1, long1, lat2, long2):
    """
    Calculates the haversine distance between 2 sets of GPS coordinates in df
    """
    r = 6371
    phi1 = np.radians(df[lat1])
    phi2 = np.radians(df[lat2])
    
    delta_phi = np.radians(df[lat2]-df[lat1])
    delta_lambda = np.radians(df[long2]-df[long1])
    
    a = np.sin(delta_phi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(delta_lambda/2)**2
    c = 2*np.arctan2(np.sqrt(a), np.sqrt(1-a))
    d = (r*c) # in kilometers
    return d


In [7]:
df['dist_km'] = haversine_distance(df,'pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude')   

In [8]:
df.head()

,pickup_datetime,fare_amount,fare_class,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,dist_km
0,2010-04-19 08:17:56 UTC,6.5,0,-73.992365,40.730521,-73.975499,40.744746,1,2.126312
1,2010-04-17 15:43:53 UTC,6.9,0,-73.990078,40.740558,-73.974232,40.744114,1,1.392307
2,2010-04-17 11:23:26 UTC,10.1,1,-73.994149,40.751118,-73.960064,40.766235,2,3.326763
3,2010-04-11 21:25:03 UTC,8.9,0,-73.990485,40.756422,-73.971205,40.748192,1,1.864129
4,2010-04-17 02:19:01 UTC,19.7,1,-73.990976,40.734202,-73.905956,40.743115,1,7.231321


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   pickup_datetime    120000 non-null  object 
 1   fare_amount        120000 non-null  float64
 2   fare_class         120000 non-null  int64  
 3   pickup_longitude   120000 non-null  float64
 4   pickup_latitude    120000 non-null  float64
 5   dropoff_longitude  120000 non-null  float64
 6   dropoff_latitude   120000 non-null  float64
 7   passenger_count    120000 non-null  int64  
 8   dist_km            120000 non-null  float64
dtypes: float64(6), int64(2), object(1)
memory usage: 8.2+ MB


In [10]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   pickup_datetime    120000 non-null  datetime64[ns, UTC]
 1   fare_amount        120000 non-null  float64            
 2   fare_class         120000 non-null  int64              
 3   pickup_longitude   120000 non-null  float64            
 4   pickup_latitude    120000 non-null  float64            
 5   dropoff_longitude  120000 non-null  float64            
 6   dropoff_latitude   120000 non-null  float64            
 7   passenger_count    120000 non-null  int64              
 8   dist_km            120000 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(6), int64(2)
memory usage: 8.2 MB


In [11]:
my_time = df['pickup_datetime'][0]

In [12]:
my_time.hour

8

In [13]:
df['EDTdate'] = df['pickup_datetime'] - pd.Timedelta(hours=4)
df['Hour'] = df['EDTdate'].dt.hour
df['AMorPM'] = np.where(df['Hour']<12,'am','pm')
df['Weekday'] = df['EDTdate'].dt.strftime("%a")
df.head()


,pickup_datetime,fare_amount,fare_class,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,dist_km,EDTdate,Hour,AMorPM,Weekday
0,2010-04-19 08:17:56+00:00,6.5,0,-73.992365,40.730521,-73.975499,40.744746,1,2.126312,2010-04-19 04:17:56+00:00,4,am,Mon
1,2010-04-17 15:43:53+00:00,6.9,0,-73.990078,40.740558,-73.974232,40.744114,1,1.392307,2010-04-17 11:43:53+00:00,11,am,Sat
2,2010-04-17 11:23:26+00:00,10.1,1,-73.994149,40.751118,-73.960064,40.766235,2,3.326763,2010-04-17 07:23:26+00:00,7,am,Sat
3,2010-04-11 21:25:03+00:00,8.9,0,-73.990485,40.756422,-73.971205,40.748192,1,1.864129,2010-04-11 17:25:03+00:00,17,pm,Sun
4,2010-04-17 02:19:01+00:00,19.7,1,-73.990976,40.734202,-73.905956,40.743115,1,7.231321,2010-04-16 22:19:01+00:00,22,pm,Fri


In [14]:
cat_cols = ['Hour', 'AMorPM', 'Weekday']
cont_cols = ['pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude', 'passenger_count', 'dist_km']
y_col = ['fare_amount']


In [15]:
df.dtypes

pickup_datetime      datetime64[ns, UTC]
fare_amount                      float64
fare_class                         int64
pickup_longitude                 float64
pickup_latitude                  float64
dropoff_longitude                float64
dropoff_latitude                 float64
passenger_count                    int64
dist_km                          float64
EDTdate              datetime64[ns, UTC]
Hour                               int32
AMorPM                            object
Weekday                           object
dtype: object

In [16]:
for cat in cat_cols:
    df[cat] = df[cat].astype('category')

In [17]:
df.dtypes

pickup_datetime      datetime64[ns, UTC]
fare_amount                      float64
fare_class                         int64
pickup_longitude                 float64
pickup_latitude                  float64
dropoff_longitude                float64
dropoff_latitude                 float64
passenger_count                    int64
dist_km                          float64
EDTdate              datetime64[ns, UTC]
Hour                            category
AMorPM                          category
Weekday                         category
dtype: object

In [18]:
df['Hour'].head()

0     4
1    11
2     7
3    17
4    22
Name: Hour, dtype: category
Categories (24, int32): [0, 1, 2, 3, ..., 20, 21, 22, 23]

In [19]:
df['AMorPM'].head()

0    am
1    am
2    am
3    pm
4    pm
Name: AMorPM, dtype: category
Categories (2, object): ['am', 'pm']

In [20]:
df['Weekday'].head()

0    Mon
1    Sat
2    Sat
3    Sun
4    Fri
Name: Weekday, dtype: category
Categories (7, object): ['Fri', 'Mon', 'Sat', 'Sun', 'Thu', 'Tue', 'Wed']

In [21]:
df["Weekday"].cat.codes # This is how the computer sees the data. It converts the data into numbers

0         1
1         2
2         2
3         3
4         0
         ..
119995    3
119996    0
119997    3
119998    5
119999    2
Length: 120000, dtype: int8

In [22]:
df['Weekday'].cat.codes.values # Now we have the data in a numpy array

array([1, 2, 2, ..., 3, 5, 2], dtype=int8)

In [23]:
hr = df['Hour'].cat.codes.values
ampm = df['AMorPM'].cat.codes.values
wkdy = df['Weekday'].cat.codes.values

In [24]:
hr

array([ 4, 11,  7, ..., 14,  4, 12], dtype=int8)

In [25]:
cats = np.stack([hr, ampm, wkdy], 1)
cats

array([[ 4,  0,  1],
       [11,  0,  2],
       [ 7,  0,  2],
       ...,
       [14,  1,  3],
       [ 4,  0,  5],
       [12,  1,  2]], dtype=int8)

In [26]:
cats = torch.tensor(cats, dtype=torch.int64)
cats

tensor([[ 4,  0,  1],
        [11,  0,  2],
        [ 7,  0,  2],
        ...,
        [14,  1,  3],
        [ 4,  0,  5],
        [12,  1,  2]])

In [27]:
conts = np.stack([df[col].values for col in cont_cols], 1)
conts

array([[ 40.730521  , -73.992365  ,  40.744746  , -73.975499  ,
          1.        ,   2.12631159],
       [ 40.740558  , -73.990078  ,  40.744114  , -73.974232  ,
          1.        ,   1.39230687],
       [ 40.751118  , -73.994149  ,  40.766235  , -73.960064  ,
          2.        ,   3.32676344],
       ...,
       [ 40.749772  , -73.988574  ,  40.707799  , -74.011541  ,
          3.        ,   5.05252282],
       [ 40.724529  , -74.004449  ,  40.730765  , -73.992697  ,
          1.        ,   1.20892296],
       [ 40.77192   , -73.955415  ,  40.763015  , -73.967623  ,
          3.        ,   1.42739869]])

In [28]:
conts = torch.tensor(conts, dtype=torch.float)
conts

tensor([[ 40.7305, -73.9924,  40.7447, -73.9755,   1.0000,   2.1263],
        [ 40.7406, -73.9901,  40.7441, -73.9742,   1.0000,   1.3923],
        [ 40.7511, -73.9941,  40.7662, -73.9601,   2.0000,   3.3268],
        ...,
        [ 40.7498, -73.9886,  40.7078, -74.0115,   3.0000,   5.0525],
        [ 40.7245, -74.0044,  40.7308, -73.9927,   1.0000,   1.2089],
        [ 40.7719, -73.9554,  40.7630, -73.9676,   3.0000,   1.4274]])

In [29]:
y = torch.tensor(df[y_col].values, dtype=torch.float)

In [30]:
cats.shape

torch.Size([120000, 3])

In [31]:
conts.shape

torch.Size([120000, 6])

In [32]:
y.shape

torch.Size([120000, 1])

In [33]:
cat_szs = [len(df[col].cat.categories) for col in cat_cols]
cat_szs

[24, 2, 7]

In [34]:
emb_szs = [(size, min(50, (size+1)//2)) for size in cat_szs]
emb_szs

[(24, 12), (2, 1), (7, 4)]

In [35]:
catz = cats[:4]

In [36]:
catz

tensor([[ 4,  0,  1],
        [11,  0,  2],
        [ 7,  0,  2],
        [17,  1,  3]])

In [37]:
selfembeds = nn.ModuleList([nn.Embedding(ni,nf) for ni,nf in emb_szs])
selfembeds

ModuleList(
  (0): Embedding(24, 12)
  (1): Embedding(2, 1)
  (2): Embedding(7, 4)
)

In [38]:
# Forward method
embeddingz = []

for i,e in enumerate(selfembeds):
    embeddingz.append(e(catz[:,i]))

embeddingz

[tensor([[ 0.6115, -0.0547,  1.5486, -1.4850, -0.9014,  0.5312,  0.0199,  1.0404,
          -0.7837,  0.2427,  0.6457,  1.0947],
         [ 2.5074, -0.6761, -0.1893,  0.0671, -1.4539, -0.8383,  0.3955, -0.5659,
           1.0410,  0.3227, -0.0678, -0.7484],
         [ 1.1847, -0.6428, -0.8680,  0.8897, -0.9735, -0.7686, -2.3269, -0.7322,
           0.7668, -1.4281, -0.2246,  2.3676],
         [ 0.1743,  0.7967, -0.7266, -0.4069,  0.3656, -0.4772, -1.4098,  0.7624,
          -0.5165, -1.3842, -0.8585, -0.2707]], grad_fn=<EmbeddingBackward0>),
 tensor([[0.6309],
         [0.6309],
         [0.6309],
         [0.4540]], grad_fn=<EmbeddingBackward0>),
 tensor([[-0.3622,  0.5979, -0.8351,  0.7469],
         [ 0.2854,  0.1752,  0.3870, -0.0201],
         [ 0.2854,  0.1752,  0.3870, -0.0201],
         [-1.1664, -0.9692, -1.1580,  1.0681]], grad_fn=<EmbeddingBackward0>)]

In [39]:
z = torch.cat(embeddingz, 1)
z



tensor([[ 0.6115, -0.0547,  1.5486, -1.4850, -0.9014,  0.5312,  0.0199,  1.0404,
         -0.7837,  0.2427,  0.6457,  1.0947,  0.6309, -0.3622,  0.5979, -0.8351,
          0.7469],
        [ 2.5074, -0.6761, -0.1893,  0.0671, -1.4539, -0.8383,  0.3955, -0.5659,
          1.0410,  0.3227, -0.0678, -0.7484,  0.6309,  0.2854,  0.1752,  0.3870,
         -0.0201],
        [ 1.1847, -0.6428, -0.8680,  0.8897, -0.9735, -0.7686, -2.3269, -0.7322,
          0.7668, -1.4281, -0.2246,  2.3676,  0.6309,  0.2854,  0.1752,  0.3870,
         -0.0201],
        [ 0.1743,  0.7967, -0.7266, -0.4069,  0.3656, -0.4772, -1.4098,  0.7624,
         -0.5165, -1.3842, -0.8585, -0.2707,  0.4540, -1.1664, -0.9692, -1.1580,
          1.0681]], grad_fn=<CatBackward0>)

In [40]:
selfembeddrop = nn.Dropout(0.4)

In [41]:
z = selfembeddrop(z)

In [42]:
z

tensor([[ 1.0192, -0.0911,  2.5810, -0.0000, -0.0000,  0.8853,  0.0000,  1.7339,
         -1.3061,  0.4044,  0.0000,  1.8245,  1.0515, -0.0000,  0.0000, -1.3919,
          0.0000],
        [ 4.1789, -0.0000, -0.0000,  0.0000, -0.0000, -1.3971,  0.6591, -0.0000,
          1.7349,  0.5378, -0.1129, -0.0000,  1.0515,  0.0000,  0.2920,  0.6450,
         -0.0334],
        [ 1.9744, -1.0713, -0.0000,  1.4828, -0.0000, -1.2810, -3.8782, -1.2203,
          1.2780, -2.3802, -0.3744,  3.9460,  1.0515,  0.4757,  0.2920,  0.6450,
         -0.0334],
        [ 0.2905,  1.3279, -1.2110, -0.0000,  0.6093, -0.7954, -2.3497,  1.2707,
         -0.8608, -2.3071, -1.4308, -0.0000,  0.0000, -0.0000, -1.6153, -1.9300,
          0.0000]], grad_fn=<MulBackward0>)

In [43]:
class TabularModel(nn.Module):
    
    def __init__(self, emb_szs, n_cont, out_sz, layers, p=0.5):
        super().__init__()
        self.embeds = nn.ModuleList([nn.Embedding(ni, nf) for ni,nf in emb_szs])
        self.emb_drop = nn.Dropout(p)
        self.bn_cont = nn.BatchNorm1d(n_cont)
        
        layerlist = []
        n_emb = sum((nf for ni,nf in emb_szs))
        n_in = n_emb + n_cont
        
        for i in layers:
            layerlist.append(nn.Linear(n_in,i)) 
            layerlist.append(nn.ReLU(inplace=True))
            layerlist.append(nn.BatchNorm1d(i))
            layerlist.append(nn.Dropout(p))
            n_in = i
        layerlist.append(nn.Linear(layers[-1],out_sz))
            
        self.layers = nn.Sequential(*layerlist)
    
    def forward(self, x_cat, x_cont):
        embeddings = []
        for i,e in enumerate(self.embeds):
            embeddings.append(e(x_cat[:,i]))
        x = torch.cat(embeddings, 1)
        x = self.emb_drop(x)
        
        x_cont = self.bn_cont(x_cont)
        x = torch.cat([x, x_cont], 1)
        x = self.layers(x)
        return x

In [45]:
torch.manual_seed(33)
model = TabularModel(emb_szs, conts.shape[1], 1, [200,100], p=0.4)


In [46]:
model

TabularModel(
  (embeds): ModuleList(
    (0): Embedding(24, 12)
    (1): Embedding(2, 1)
    (2): Embedding(7, 4)
  )
  (emb_drop): Dropout(p=0.4, inplace=False)
  (bn_cont): BatchNorm1d(6, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layers): Sequential(
    (0): Linear(in_features=23, out_features=200, bias=True)
    (1): ReLU(inplace=True)
    (2): BatchNorm1d(200, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=200, out_features=100, bias=True)
    (5): ReLU(inplace=True)
    (6): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Dropout(p=0.4, inplace=False)
    (8): Linear(in_features=100, out_features=1, bias=True)
  )
)

In [47]:
criterion = nn.MSELoss() # For regression problems
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [48]:
batch_size = 60000
test_size = int(batch_size * .2)    # 20%

# Data is already shuffled
cat_train = cats[:batch_size-test_size]
cat_test = cats[batch_size-test_size:batch_size]
con_train = conts[:batch_size-test_size]
con_test = conts[batch_size-test_size:batch_size]

# y_train and y_test are the same as con_train and con_test, just for the target
y_train = y[:batch_size-test_size]
y_test = y[batch_size-test_size:batch_size]


In [49]:
len(cat_train)

48000

In [50]:
len(con_train)

48000

In [51]:
len(cat_test)

12000

In [ ]:
import time
start_time = time.time()

epochs = 300
losses = []

for i in range(epochs):
    i+=1
    y_pred = model(cat_train, con_train)
    loss = torch.sqrt(criterion(y_pred, y_train)) # RMSE
    losses.append(loss)
    
    # a neat trick to save screen space:
    if i%10 == 1:
        print(f'epoch: {i:3}  loss: {loss.item():10.8f}')

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f'epoch: {i:3}  loss: {loss.item():10.8f}') # print the last line
print(f'\nDuration: {time.time() - start_time:.0f} seconds') # print the time elapsed

epoch:   1  loss: 3.59641385
epoch:  26  loss: 3.59870386
epoch:  51  loss: 3.53037834
epoch:  76  loss: 3.51018429
epoch: 101  loss: 3.45336580
epoch: 126  loss: 3.42833161
epoch: 151  loss: 3.41817117
epoch: 176  loss: 3.37246442
epoch: 201  loss: 3.36571693
epoch: 226  loss: 3.31376839
epoch: 251  loss: 3.32105970
epoch: 276  loss: 3.30253959
epoch: 300  loss: 3.28984857

Duration: 42 seconds


In [ ]:


with torch.no_grad():
    y_val = model(cat_test, con_test)
    loss = torch.sqrt(criterion(y_val, y_test))
print(f'RMSE: {loss:.8f}')

RMSE: 3.20639586


In [59]:
for i in range(10):
    diff = np.abs(y_val[i].item() - y_test[i].item())
    print(f'{i+1:2}. predicted: {y_val[i].item():8.2f} actual: {y_test[i].item():8.2f} difference: {diff:8.2f}')

 1. predicted:     2.63 actual:     2.90 difference:     0.27
 2. predicted:    23.94 actual:     5.70 difference:    18.24
 3. predicted:     6.90 actual:     7.70 difference:     0.80
 4. predicted:    13.37 actual:    12.50 difference:     0.87
 5. predicted:     5.94 actual:     4.10 difference:     1.84
 6. predicted:     6.12 actual:     5.30 difference:     0.82
 7. predicted:     4.22 actual:     3.70 difference:     0.52
 8. predicted:    17.68 actual:    14.50 difference:     3.18
 9. predicted:     2.44 actual:     5.70 difference:     3.26
10. predicted:    12.35 actual:    10.10 difference:     2.25


In [60]:
torch.save(model.state_dict(), 'TaxiFareRegrModel.pt')